In [1]:
import numpy as np
print(np.__version__)

from datetime import datetime
import torch
print(torch.__version__)

import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

from tqdm.notebook import trange

import random
import math
import time

1.26.4
2.9.0


In [2]:
import numpy as np
from numba import njit

SIZE = 5
EMPTY = 0
P1 = 1
P2 = -1
max_moves = 100


def build_action_map(size):
    actions = []

    for r in range(size):
        for c in range(size):
            if not (r == 0 or r == size-1 or c == 0 or c == size-1):
                continue

            if c != 0:
                actions.append((r, c, r, 0))
            if c != size - 1:
                actions.append((r, c, r, size - 1))
            if r != 0:
                actions.append((r, c, 0, c))
            if r != size - 1:
                actions.append((r, c, size - 1, c))

    return np.array(actions, dtype=np.int8)  # (A, 4)

@njit
def get_valid_moves_numba(state, action_map):
    A = action_map.shape[0]
    valid = np.zeros(A, dtype=np.uint8)

    for i in range(A):
        r, c = action_map[i, 0], action_map[i, 1]
        piece = state[r, c]

        if piece == EMPTY or piece == P1:
            valid[i] = 1

    return valid

@njit
def push_numba(state, action):
    r, c, dr, dc = action
    size = state.shape[0]

    new = state.copy()

    if r == dr:
        # row shift
        if dc == 0:
            # push from right → shift right
            for j in range(c, 0, -1):
                new[r, j] = new[r, j-1]
            new[r, 0] = P1
        else:
            # push from left → shift left
            for j in range(c, size-1):
                new[r, j] = new[r, j+1]
            new[r, size-1] = P1

    else:
        # column shift
        if dr == 0:
            for i in range(r, 0, -1):
                new[i, c] = new[i-1, c]
            new[0, c] = P1
        else:
            for i in range(r, size-1):
                new[i, c] = new[i+1, c]
            new[size-1, c] = P1

    return new

@njit
def has_line_numba(state, player):
    size = state.shape[0]

    for i in range(size):
        if np.all(state[i, :] == player):
            return True
        if np.all(state[:, i] == player):
            return True

    diag1 = True
    diag2 = True

    for i in range(size):
        if state[i, i] != player:
            diag1 = False
        if state[i, size - 1 - i] != player:
            diag2 = False

    return diag1 or diag2

def encode_state_batch(states):
    # states: (B, S, S)

    p2 = (states == P2)
    empty = (states == EMPTY)
    p1 = (states == P1)

    encoded = np.stack((p2, empty, p1), axis=1).astype(np.float32)
    # (B, 3, S, S)

    return encoded

class QuixoFast:
    def __init__(self):
        self.size = SIZE
        self.action_map = build_action_map(SIZE)   # (A, 4)
        self.action_size = self.action_map.shape[0]

        # --- NEW: fast lookup ---
        self._action_to_index = {}
        for i in range(self.action_size):
            key = tuple(self.action_map[i])
            self._action_to_index[key] = i

    def get_initial_state(self):
        return np.zeros((self.size, self.size), dtype=np.int8)

    def change_perspective(self, state, player):
        return state * player

    def get_valid_moves(self, state):
        return get_valid_moves_numba(state, self.action_map)

    def get_next_state(self, state, action, player):
        canonical = state * player
        next_state = push_numba(canonical, self.action_map[action])
        return next_state * player

    def get_value_and_terminated(self, state, player, move_count):
        if has_line_numba(state, -player):
            return -1, True
        if has_line_numba(state, player):
            return 1, True
        if move_count >= max_moves:
            return 0, True
        return 0, False
    
    def encode_action(self, r, c, dr, dc):
        return self._action_to_index[(r, c, dr, dc)]

    def decode_action(self, action):
        return self.action_map[action]
    
    def get_encoded_state(self, state):
        """
        Encodes the state into a 3-channel representation for the ResNet.
        Channels: [Opponent's Pieces, Empty Squares, Current Player's Pieces]
        """
        # Constants used: P1 = 1, P2 = -1, EMPTY = 0
        if state.ndim == 2: # Single state (5, 5)
            return np.stack((state == -1, state == 0, state == 1)).astype(np.float32)
        
        # Batch of states (Batch, 5, 5)
        # We swap axes so the output is (Batch, 3, 5, 5)
        encoded = np.stack((state == -1, state == 0, state == 1)).astype(np.float32)
        return np.swapaxes(encoded, 0, 1)

    def get_opponent(self, player):
        """Returns the other player (-1 for 1, 1 for -1)."""
        return -player

    def get_opponent_value(self, value):
        """Flips the value for the other player's perspective."""
        return -value

In [ ]:
"""
PPO for Quixo – self-play training with evaluation vs. uniform-random.

Design notes
------------
* Both players share one network; states are always presented in canonical
  form (own pieces = +1, opponent = -1) so the network is player-agnostic.
* All intermediate rewards are 0; terminal reward is +1 (win), -1 (loss),
  0 (draw/timeout). Monte-Carlo returns are therefore trivially computable:
      G_i = gamma^(T-1-i) * r_terminal
* We collect complete episodes rather than fixed-length rollouts to avoid
  having to bootstrap mid-game values across the player boundary.
* Valid-move masks are stored in the replay buffer and applied both at
  rollout time and during the PPO gradient step, keeping the policy
  distribution restricted to legal actions throughout.
* Evaluation rotates PPO between player-1 and player-2 seats to control
  for first-mover advantage.

Paste (or import) the QuixoFast class above this file, then call train().
"""

import csv
import time
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical

# ── device ────────────────────────────────────────────────────────────────────
device = torch.device(
    "cuda"  if torch.cuda.is_available()  else
    "mps"   if torch.backends.mps.is_available() else
    "cpu"
)

# ── hyperparameters ───────────────────────────────────────────────────────────
GAMMA               = 0.99   # discount; close to 1 since reward is terminal-only
CLIP_EPS            = 0.2    # PPO clipping radius
LR                  = 3e-4
VALUE_COEF          = 0.5    # weight of value loss relative to policy loss
ENTROPY_COEF        = 0.01   # entropy bonus; keeps exploration alive
N_EPOCHS            = 4      # passes over each collected buffer
BATCH_SIZE          = 256
GRAD_CLIP           = 0.5    # max gradient norm
EPISODES_PER_UPDATE = 128    # self-play games collected before each PPO step
N_UPDATES           = 500
EVAL_INTERVAL       = 10     # evaluate every N updates
N_EVAL_GAMES        = 200    # games per evaluation (split evenly across sides)


# ── network ───────────────────────────────────────────────────────────────────
class ResBlock(nn.Module):
    def __init__(self, ch: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch, ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(ch),
        )

    def forward(self, x):
        return F.relu(x + self.net(x), inplace=True)


class QuixoNet(nn.Module):
    """
    Input  : (B, 3, 5, 5)  – channels are [opponent, empty, self]
    Outputs: logits (B, A),  value (B, 1) ∈ [-1, 1]
    """
    def __init__(self, action_size: int, num_hidden: int = 64, num_res: int = 4):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, num_hidden, 3, padding=1, bias=False),
            nn.BatchNorm2d(num_hidden),
            nn.ReLU(inplace=True),
        )
        self.backbone = nn.Sequential(*[ResBlock(num_hidden) for _ in range(num_res)])

        # Policy head: conv bottleneck → flat → logits
        self.policy_head = nn.Sequential(
            nn.Conv2d(num_hidden, 2, 1, bias=False),
            nn.BatchNorm2d(2),
            nn.ReLU(inplace=True),
            nn.Flatten(),
            nn.Linear(2 * 5 * 5, action_size),
        )

        # Value head: conv bottleneck → MLP → tanh scalar
        self.value_head = nn.Sequential(
            nn.Conv2d(num_hidden, 1, 1, bias=False),
            nn.BatchNorm2d(1),
            nn.ReLU(inplace=True),
            nn.Flatten(),
            nn.Linear(5 * 5, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, 1),
            nn.Tanh(),
        )

    def forward(self, x):
        x = self.backbone(self.stem(x))
        return self.policy_head(x), self.value_head(x)


# ── helpers ───────────────────────────────────────────────────────────────────
def _encode(env, canonical_state) -> torch.Tensor:
    """Return a (3,5,5) float32 tensor on device."""
    arr = env.get_encoded_state(canonical_state)   # (3,5,5) ndarray
    return torch.as_tensor(arr, dtype=torch.float32, device=device)


def _masked_dist(logits: torch.Tensor, valid: torch.Tensor) -> Categorical:
    """Categorical distribution over legal actions only."""
    return Categorical(logits=logits + (1.0 - valid) * -1e9)


def _mc_returns(terminal_reward: float, n: int, gamma: float) -> list:
    """
    Discounted returns for a trajectory whose only non-zero reward is the last.
    G_i = gamma^(n-1-i) * terminal_reward,  computed in O(n) backwards.
    """
    G = terminal_reward
    rets = [0.0] * n
    for i in range(n - 1, -1, -1):
        rets[i] = G
        G *= gamma
    return rets


# ── rollout collection ────────────────────────────────────────────────────────
def collect_episodes(env, net: QuixoNet, n_episodes: int, gamma: float) -> list:
    """
    Play n_episodes self-play games and return a flat list of transition dicts.

    Each dict contains:
        state      : FloatTensor (3,5,5) on CPU
        action     : int
        log_prob   : float   – log π_old(a|s)
        value      : float   – V_old(s)
        return_    : float   – MC return
        advantage  : float   – return_ - value
        valid_mask : ndarray (A,) uint8
    """
    net.eval()
    buffer = []

    for _ in range(n_episodes):
        state   = env.get_initial_state()
        player  = 1
        move_ct = 0
        done    = False

        traj = {1: [], -1: []}      # separate trajectory per player

        while not done:
            canonical = env.change_perspective(state, player)
            enc       = _encode(env, canonical)             # (3,5,5)
            valid_np  = env.get_valid_moves(canonical)      # (A,) uint8
            valid_t   = torch.as_tensor(valid_np, dtype=torch.float32, device=device)

            with torch.no_grad():
                logits, value = net(enc.unsqueeze(0))       # (1,A), (1,1)

            dist   = _masked_dist(logits.squeeze(0), valid_t)
            action = dist.sample()
            lp     = dist.log_prob(action)

            next_state     = env.get_next_state(state, action.item(), player)
            move_ct       += 1
            reward, done   = env.get_value_and_terminated(next_state, player, move_ct)

            traj[player].append({
                "state"     : enc.cpu(),
                "action"    : action.item(),
                "log_prob"  : lp.item(),
                "value"     : value.item(),
                "valid_mask": valid_np,
            })

            state  = next_state
            player = -player   # swap sides

        # After the loop, player was already flipped; undo to recover last mover.
        last_mover = -player
        # Zero-sum: last mover gets `reward`, opponent gets `-reward`.
        r = {last_mover: float(reward), -last_mover: -float(reward)}

        for p in (1, -1):
            t = traj[p]
            if not t:
                continue
            mc = _mc_returns(r[p], len(t), gamma)
            for step, ret in zip(t, mc):
                step["return_"]   = ret
                step["advantage"] = ret - step["value"]
                buffer.append(step)

    return buffer


# ── PPO gradient step ─────────────────────────────────────────────────────────
def ppo_update(
    net, optimizer, buffer,
    n_epochs, batch_size, clip_eps, value_coef, ent_coef, grad_clip,
) -> dict:
    net.train()

    N = len(buffer)

    # Pre-stack everything once to avoid repeated small GPU transfers.
    states      = torch.stack([t["state"]      for t in buffer]).to(device)          # (N,3,5,5)
    actions     = torch.tensor([t["action"]    for t in buffer],                     device=device)
    old_lps     = torch.tensor([t["log_prob"]  for t in buffer], dtype=torch.float32, device=device)
    returns     = torch.tensor([t["return_"]   for t in buffer], dtype=torch.float32, device=device)
    advantages  = torch.tensor([t["advantage"] for t in buffer], dtype=torch.float32, device=device)
    valid_masks = torch.tensor(
        np.stack([t["valid_mask"] for t in buffer]), dtype=torch.float32, device=device
    )  # (N, A)

    # Normalise advantages globally so gradient magnitudes stay stable.
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    totals = defaultdict(float)
    n_batches = 0

    for _ in range(n_epochs):
        perm = torch.randperm(N, device=device)
        for start in range(0, N - batch_size + 1, batch_size):
            idx = perm[start : start + batch_size]

            logits, values = net(states[idx])

            dist    = _masked_dist(logits, valid_masks[idx])
            new_lp  = dist.log_prob(actions[idx])
            entropy = dist.entropy().mean()

            # Clipped surrogate objective
            ratio   = torch.exp(new_lp - old_lps[idx])
            adv     = advantages[idx]
            policy_loss = -torch.min(
                ratio * adv,
                torch.clamp(ratio, 1.0 - clip_eps, 1.0 + clip_eps) * adv,
            ).mean()

            value_loss = F.mse_loss(values.squeeze(-1), returns[idx])

            loss = policy_loss + value_coef * value_loss - ent_coef * entropy

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), grad_clip)
            optimizer.step()

            totals["policy_loss"] += policy_loss.item()
            totals["value_loss"]  += value_loss.item()
            totals["entropy"]     += entropy.item()
            n_batches             += 1

    return {k: v / n_batches for k, v in totals.items()}


# ── evaluation ────────────────────────────────────────────────────────────────
def evaluate_vs_random(env, net: QuixoNet, n_games: int = 200) -> dict:
    """
    Pit the learned policy against uniform-random-over-legal-moves.
    PPO plays as player 1 for the first half and player -1 for the second
    half to control for first-mover advantage.

    Returns win / draw / loss rates from PPO's perspective.
    """
    net.eval()
    counts = {"win": 0, "draw": 0, "loss": 0}

    for g in range(n_games):
        ppo_side = 1 if g < n_games // 2 else -1
        state    = env.get_initial_state()
        player   = 1
        move_ct  = 0
        done     = False

        while not done:
            canonical = env.change_perspective(state, player)
            valid_np  = env.get_valid_moves(canonical)

            if player == ppo_side:
                enc    = _encode(env, canonical).unsqueeze(0)
                with torch.no_grad():
                    logits, _ = net(enc)
                valid_t = torch.as_tensor(valid_np, dtype=torch.float32, device=device)
                action  = _masked_dist(logits.squeeze(0), valid_t).sample().item()
            else:
                action = int(np.random.choice(np.where(valid_np)[0]))

            next_state    = env.get_next_state(state, action, player)
            move_ct      += 1
            reward, done  = env.get_value_and_terminated(next_state, player, move_ct)

            if done:
                ppo_reward = reward if player == ppo_side else -reward
                if   ppo_reward > 0: counts["win"]  += 1
                elif ppo_reward < 0: counts["loss"] += 1
                else:                counts["draw"] += 1

            state  = next_state
            player = -player

    return {k: v / n_games for k, v in counts.items()}


# ── training loop ─────────────────────────────────────────────────────────────
def train(
    n_updates           = N_UPDATES,
    episodes_per_update = EPISODES_PER_UPDATE,
    eval_interval       = EVAL_INTERVAL,
    n_eval_games        = N_EVAL_GAMES,
):
    env = QuixoFast()
    net = QuixoNet(action_size=env.action_size).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=LR)

    print(f"Device      : {device}")
    print(f"Action size : {env.action_size}")
    print(f"Parameters  : {sum(p.numel() for p in net.parameters()):,}")
    print()

    header = (
        f"{'Upd':>5}  "
        f"{'Win':>6} {'Draw':>6} {'Loss':>6}  "
        f"{'π-loss':>8} {'V-loss':>8} {'H':>7}  "
        f"{'Buf':>5}  {'s':>5}"
    )
    print(header)
    print("─" * len(header))

    log = defaultdict(list)   # accumulates everything for later plotting/analysis

    for upd in range(1, n_updates + 1):
        t0 = time.perf_counter()

        buffer  = collect_episodes(env, net, episodes_per_update, GAMMA)
        metrics = ppo_update(
            net, opt, buffer,
            N_EPOCHS, BATCH_SIZE, CLIP_EPS, VALUE_COEF, ENTROPY_COEF, GRAD_CLIP,
        )

        dt = time.perf_counter() - t0

        log["update"].append(upd)
        log["policy_loss"].append(metrics["policy_loss"])
        log["value_loss"].append(metrics["value_loss"])
        log["entropy"].append(metrics["entropy"])
        log["buffer_size"].append(len(buffer))

        # Evaluate and log every update (or use run frequency via eval_interval)
        if upd % eval_interval == 0:
            rates = evaluate_vs_random(env, net, n_eval_games)
        else:
            rates = evaluate_vs_random(env, net, max(10, n_eval_games // 10))

        log["eval_update"].append(upd)
        log["win_rate"].append(rates["win"])
        log["draw_rate"].append(rates["draw"])
        log["loss_rate"].append(rates["loss"])

        eval_cols = f"{rates['win']:6.1%} {rates['draw']:6.1%} {rates['loss']:6.1%}"

        # Persist log to file after each update
        log_row = [
            upd,
            rates["win"],
            rates["draw"],
            rates["loss"],
            metrics["policy_loss"],
            metrics["value_loss"],
            metrics["entropy"],
            len(buffer),
            dt,
        ]
        write_header = (upd == 1)
        with open("ppo_training_log.csv", "a", newline="") as f:
            writer = csv.writer(f)
            if write_header:
                writer.writerow([
                    "update",
                    "win_rate",
                    "draw_rate",
                    "loss_rate",
                    "policy_loss",
                    "value_loss",
                    "entropy",
                    "buffer_size",
                    "seconds",
                ])
            writer.writerow(log_row)

        print(
            f"{upd:5d}  "
            f"{eval_cols}  "
            f"{metrics['policy_loss']:8.4f} {metrics['value_loss']:8.4f} "
            f"{metrics['entropy']:7.4f}  "
            f"{len(buffer):5d}  {dt:5.1f}"
        )

    return net, dict(log)


if __name__ == "__main__":
    net, log = train()

Device      : mps
Action size : 44
Parameters  : 301,963

  Upd     Win   Draw   Loss    π-loss   V-loss       H    Buf      s
────────────────────────────────────────────────────────────────────
    1                           -0.0492   0.5518  3.3089   5881   49.9
    2                           -0.0638   0.4958  3.3035   5581   36.6
    3                           -0.0640   0.4653  3.2831   5802   36.7


KeyboardInterrupt: 